# ESP-Lab workflow figure gallery

Discover the figures that actually exist in the workflow output directory and build a searchable, responsive webpage. Only supported image files matching `fig_*` are included. Existing titles and captions are retained, stale manifest entries are removed, and metadata is generated for newly discovered figures.

In [5]:
# ============================================================
# Configuration
# ============================================================
from pathlib import Path

DIAG_DIR = Path("/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag")
FIGURE_PATTERN = "fig_*"
PORTAL_URL = "https://portal.nersc.gov/cfs/e3sm/zhan391/esp-lab_diag/index.html"

In [6]:
# Imports
import sys

import pandas as pd
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "esp_lab").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from esp_lab.diagnostics.web import (
    discover_workflow_figures,
    generate_diagnostics_webpage,
)

## 1. Discover workflow figures

Scan the directory rather than trusting old manifest entries. The synchronized catalog below is exactly what the webpage will show.

In [7]:
manifest = discover_workflow_figures(
    DIAG_DIR,
    pattern=FIGURE_PATTERN,
    write_manifest=True,
)
catalog = pd.DataFrame(manifest["figures"])[
    ["file", "group", "mode", "metric", "title", "caption"]
]
print(f"Found {len(catalog)} workflow figures in {DIAG_DIR}")
display(catalog.groupby("group").size().rename("figures").to_frame())
display(catalog)

Found 138 workflow figures in /global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag


,figures
group,
ELI,6
LEAD_ACC,13
LEAD_RMSE,21
MOV,48
SST_INDEX,44
TC,6


,file,group,mode,metric,title,caption
0,fig_amo_eof_patterns.png,MOV,AMO,eof_patterns,AMO EOF Patterns,"Y1-Y2 seasons: DJF/MAM/JJA/SON, HADISST2 refer..."
1,fig_amo_global_teleconnection_patterns.png,MOV,AMO,global_teleconnection_patterns,Global AMO SST Teleconnection Patterns,"Full-map SST regression patterns onto AMO PCs,..."
2,fig_amo_pc_time_series.png,MOV,AMO,pc_time_series,AMO Projected EOF PC Time Series,Projected EOF PC time series for hindcasts ini...
3,fig_amo_skill.png,MOV,AMO,skill,Seasonal AMO Skill,ACC and NRMSE using lead-specific common targe...
4,fig_atlmdr_atlmdr_acc_skill.png,SST_INDEX,,atlmdr_sst_skill,AtlMDR SST Skill,Common initialization-year verification cohort...
...,...,...,...,...,...,...
133,fig_trefht_rmse_compare_global.png,LEAD_RMSE,,rmse_compare,RMSE Comparison: TREFHT,RMSE comparison across experiments
134,fig_trefht_rmse_diff_compare.png,LEAD_RMSE,,rmse_compare,RMSE Comparison: TREFHT,RMSE comparison across experiments
135,fig_trefht_rmse_diff_global.png,LEAD_RMSE,,rmse_compare,RMSE Comparison: TREFHT,RMSE comparison across experiments
136,fig_tsa_tsa_acc_skill.png,SST_INDEX,,tsa_sst_skill,TSA SST Skill,Common initialization-year verification cohort...


## 2. Build the webpage

In [8]:
output_html = generate_diagnostics_webpage(
    DIAG_DIR,
    discover_figures=True,
    figure_pattern=FIGURE_PATTERN,
)
print(f"Webpage created with {len(catalog)} figures: {output_html}")
display(HTML(f'<a href="{PORTAL_URL}" target="_blank">Open the ESP-Lab figure gallery</a>'))

Webpage created with 138 figures: /global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag/index.html


The resulting `index.html` provides group and metric filters, text search, responsive figure cards, full-resolution viewing, and figure comparison. Generation also makes the cataloged figures publicly readable and the gallery directory publicly traversable so the portal can serve every image. Re-run the two sections whenever workflow figures are added, replaced, or removed.